# Fixed-Extractor Prompt Technique and Deep Behavior Analysis

This notebook reruns the main prompt-technique analysis and the deeper behavior-control analysis using the corrected result files:

- `results/gemma2-9b_fixextractor.jsonl`
- `results/llama3.1-8b_fixextractor.jsonl`
- `results/qwen2.5-7b_fixextractor.jsonl`

Why this notebook exists:

The original extractor only accepted fenced markdown code blocks. Some Llama 3.1 responses were plain Python code without fences, so they were incorrectly marked as `No code extracted`. The fixed extractor falls back to the full response and validates it with `ast.parse()`.

The goal is to answer two questions:

1. How do the main prompt-technique results change after fixing extraction?
2. Does the tone effect still disappear after controlling for output length, code-only behavior, extraction/parsing validity, and runtime?

## 1. Imports and Paths

All outputs from this notebook are written to:

`analysis_outputs/fixed_extractor/`

The new report is written to:

`prompt_technique_effect_report_fixextractor.md`

In [ ]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd()
PROBLEMS_PATH = ROOT / "problems.csv"
PROMPT_MATRIX_PATH = ROOT / "prompt_matrix_prompt_template.08.csv"
RESULTS_DIR = ROOT / "results"
OUTPUT_DIR = ROOT / "analysis_outputs" / "fixed_extractor"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = ROOT / "prompt_technique_effect_report_fixextractor.md"

print("ROOT:", ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 2. Load Fixed-Extractor Results

The result files are joined with:

- prompt-template metadata from `prompt_matrix_prompt_template.08.csv`
- problem/test metadata from `problems.csv`

Each result `task_id` is parsed as `problem_id_template_id`.

In [ ]:
def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} line {line_no}: {exc}") from exc
            row["source_file"] = path.name
            rows.append(row)
    return pd.DataFrame(rows)


def parse_result_task_id(value: object) -> tuple[int, int]:
    match = re.fullmatch(r"(\d+)_(\d+)", str(value))
    if not match:
        raise ValueError(f"Unexpected result task_id format: {value!r}")
    return int(match.group(1)), int(match.group(2))


def short_model_name(source_file: str, model_value: object = None) -> str:
    name = source_file.removesuffix("_fixextractor.jsonl")
    name = name.removesuffix("_final.jsonl")
    if name != source_file:
        return name
    if model_value is not None:
        return str(model_value).replace(":", "-")
    return source_file.removesuffix(".jsonl")


problems = pd.read_csv(PROBLEMS_PATH)
prompt_matrix = pd.read_csv(PROMPT_MATRIX_PATH).rename(columns={"ID": "template_id", "Role framing": "Role_framing"})
result_paths = sorted(RESULTS_DIR.glob("*_fixextractor.jsonl"))

if not result_paths:
    raise FileNotFoundError(f"No *_fixextractor.jsonl files found in {RESULTS_DIR}")

results = pd.concat([read_jsonl(path) for path in result_paths], ignore_index=True)
parsed_ids = results["task_id"].map(parse_result_task_id)
results["problem_id"] = parsed_ids.map(lambda pair: pair[0])
results["template_id"] = parsed_ids.map(lambda pair: pair[1])
results["model_short"] = [
    short_model_name(src, model)
    for src, model in zip(results["source_file"], results.get("model", pd.Series([None] * len(results))))
]

problem_meta = (
    problems.rename(columns={"task_id": "problem_id"})
    .sort_values(["problem_id", "category_id"])
    .drop_duplicates("problem_id")
    [["problem_id", "prompt", "test_cases", "challenge_test_list"]]
)

df = (
    results.merge(prompt_matrix, on="template_id", how="left", validate="many_to_one")
    .merge(problem_meta, on="problem_id", how="left", validate="many_to_one", suffixes=("", "_problem"))
)

print("Result files:", [p.name for p in result_paths])
print("Rows:", len(df))
print("Models:", sorted(df["model_short"].unique()))
print("Problems:", df["problem_id"].nunique())
print("Templates:", df["template_id"].nunique())
df.head(3)

## 3. Derived Metrics

Performance metrics:

- `passed_int`: exact pass/fail as 0/1
- `tests_fraction`: fraction of unit tests passed

Behavior metrics:

- `completion_tokens`: output length
- `duration_seconds`: runtime
- `likely_code_only`: response mostly consists of extracted code
- `invalid_python_after_fix`: fixed extractor produced text that is not valid Python
- `extra_text_ratio`: share of response outside extracted code
- `fixed_extractor_changed`: whether this row had different extracted code after repair

In [ ]:
def safe_len(value: object) -> int:
    if isinstance(value, str):
        return len(value)
    if pd.isna(value):
        return 0
    return len(str(value))


def count_lines(value: object) -> int:
    if not isinstance(value, str) or not value.strip():
        return 0
    return len(value.splitlines())


def error_text(errors: object) -> str:
    if isinstance(errors, list):
        return "\n".join(map(str, errors))
    if pd.isna(errors):
        return ""
    return str(errors)


def classify_error(errors: object, extracted_code: object) -> str:
    text = error_text(errors).lower()
    code = extracted_code if isinstance(extracted_code, str) else ""
    if "response is not valid python" in text:
        return "invalid_python_response"
    if not code.strip():
        return "missing_or_empty_code"
    if not text.strip():
        return "none"
    if "not defined" in text:
        return "name_not_defined"
    if "syntax" in text or "invalid syntax" in text:
        return "syntax_error"
    if "timeout" in text or "timed out" in text:
        return "timeout"
    if "importerror" in text or "no module named" in text:
        return "import_error"
    if "assert" in text or "test failed" in text:
        return "wrong_answer"
    return "other_runtime"


df["passed_int"] = df["passed"].astype(bool).astype(int)
df["tests_passed"] = pd.to_numeric(df["tests_passed"], errors="coerce")
df["tests_total"] = pd.to_numeric(df["tests_total"], errors="coerce")
df["tests_fraction"] = np.where(df["tests_total"] > 0, df["tests_passed"] / df["tests_total"], np.nan)
df["prompt_tokens"] = pd.to_numeric(df["prompt_tokens"], errors="coerce")
df["completion_tokens"] = pd.to_numeric(df["completion_tokens"], errors="coerce")
df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")
df["total_tokens"] = df["prompt_tokens"] + df["completion_tokens"]

df["response_chars"] = df["response_text"].map(safe_len)
df["code_chars"] = df["extracted_code"].map(safe_len)
df["response_lines"] = df["response_text"].map(count_lines)
df["code_lines"] = df["extracted_code"].map(count_lines)
df["has_extracted_code"] = (df["code_chars"] > 0).astype(int)
df["extra_text_chars"] = (df["response_chars"] - df["code_chars"]).clip(lower=0)
df["extra_text_ratio"] = np.where(df["response_chars"] > 0, df["extra_text_chars"] / df["response_chars"], np.nan)
df["likely_code_only"] = ((df["has_extracted_code"] == 1) & (df["extra_text_ratio"] < 0.15)).astype(int)
df["error_text"] = df["errors"].map(error_text)
df["error_category"] = [classify_error(e, c) for e, c in zip(df["errors"], df["extracted_code"])]
df["invalid_python_after_fix"] = df["error_category"].eq("invalid_python_response").astype(int)
df["runtime_failure"] = df["error_category"].isin([
    "syntax_error", "name_not_defined", "import_error", "other_runtime", "timeout", "missing_or_empty_code", "invalid_python_response"
]).astype(int)
df["log_completion_tokens"] = np.log1p(df["completion_tokens"])
df["log_duration_seconds"] = np.log1p(df["duration_seconds"])
df["fixed_extractor_changed"] = df.get("original_extracted_code_empty", pd.Series(False, index=df.index)).astype(bool).astype(int)
df["pass_changed_after_fix"] = (df.get("original_passed", df["passed"]).astype(bool) != df["passed"].astype(bool)).astype(int)

technique_candidates = ["Tone", "Pressure", "Role_framing", "Format"]
technique_factors = []
for col in technique_candidates:
    if col in df.columns and df[col].dropna().nunique() > 1:
        technique_factors.append(col)
print("Technique factors:", technique_factors)

df.to_csv(OUTPUT_DIR / "joined_fixed_extractor_data.csv", index=False)
df[["model_short", "problem_id", "template_id", *technique_factors, "passed_int", "tests_fraction", "completion_tokens", "likely_code_only", "invalid_python_after_fix", "error_category"]].head()

## 4. Extractor Fix Impact

This shows how many outputs changed because of the fixed extractor.

The key result should be that Llama 3.1 improves because many plain-code responses were previously missed by the markdown-fence-only extractor.

In [ ]:
extractor_impact = (
    df.groupby("model_short")
    .agg(
        runs=("task_id", "size"),
        original_pass_rate=("original_passed", lambda s: pd.Series(s).astype(bool).mean() if "original_passed" in df else np.nan),
        fixed_pass_rate=("passed_int", "mean"),
        original_passes=("original_passed", lambda s: int(pd.Series(s).astype(bool).sum()) if "original_passed" in df else np.nan),
        fixed_passes=("passed_int", "sum"),
        changed_extraction=("fixed_extractor_changed", "sum"),
        changed_pass=("pass_changed_after_fix", "sum"),
        invalid_python_after_fix=("invalid_python_after_fix", "sum"),
    )
    .reset_index()
)
extractor_impact.to_csv(OUTPUT_DIR / "extractor_fix_impact_by_model.csv", index=False)
extractor_impact.style.format({"original_pass_rate": "{:.1%}", "fixed_pass_rate": "{:.1%}"})

## 5. Overall Model Performance

This is the corrected version of the original model-performance table.

In [ ]:
model_summary = (
    df.groupby("model_short")
    .agg(
        runs=("task_id", "size"),
        pass_rate=("passed_int", "mean"),
        avg_tests_fraction=("tests_fraction", "mean"),
        avg_duration_seconds=("duration_seconds", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        code_only_rate=("likely_code_only", "mean"),
        invalid_python_rate=("invalid_python_after_fix", "mean"),
    )
    .reset_index()
    .sort_values("pass_rate", ascending=False)
)
model_summary.to_csv(OUTPUT_DIR / "model_summary.csv", index=False)
model_summary.style.format({
    "pass_rate": "{:.1%}", "avg_tests_fraction": "{:.1%}", "avg_duration_seconds": "{:.2f}",
    "avg_completion_tokens": "{:.0f}", "code_only_rate": "{:.1%}", "invalid_python_rate": "{:.1%}"
})

## 6. Main Prompt-Technique Summary

This reruns the original factor summaries for:

- tone
- role framing
- format

The most important table is the tone table.

In [ ]:
def mean_ci(series: pd.Series, z: float = 1.96) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").dropna()
    n = len(values)
    mean = values.mean() if n else np.nan
    se = values.std(ddof=1) / math.sqrt(n) if n > 1 else np.nan
    return pd.Series({"mean": mean, "ci_low": mean - z * se if n > 1 else np.nan, "ci_high": mean + z * se if n > 1 else np.nan, "n": n})

summary_metrics = ["passed_int", "tests_fraction", "duration_seconds", "completion_tokens", "likely_code_only", "extra_text_ratio", "invalid_python_after_fix"]
factor_summaries = []
for factor in technique_factors:
    for metric in summary_metrics:
        part = df.groupby(factor, dropna=False)[metric].apply(mean_ci).unstack().reset_index()
        part.insert(0, "factor", factor)
        part.insert(2, "metric", metric)
        factor_summaries.append(part)
factor_summary = pd.concat(factor_summaries, ignore_index=True)
factor_summary.to_csv(OUTPUT_DIR / "factor_summary.csv", index=False)

for factor in technique_factors:
    print(f"\n=== {factor} ===")
    display(factor_summary[factor_summary["factor"].eq(factor)].style.format({"mean": "{:.4f}", "ci_low": "{:.4f}", "ci_high": "{:.4f}"}))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_specs = [
    ("tests_fraction", "Average test fraction"),
    ("completion_tokens", "Completion tokens"),
    ("duration_seconds", "Duration seconds"),
    ("likely_code_only", "Code-only rate"),
]
for ax, (metric, title) in zip(axes.flat, plot_specs):
    sns.barplot(data=df, x="Tone", y=metric, errorbar=("ci", 95), ax=ax, order=["neutral", "polite", "aggressive"])
    ax.set_title(title)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 7. Paired Prompt Effects

This compares factor levels within the same `(model, problem)` pair.

For tone, the reference level is the first template order, which is neutral. So polite/aggressive deltas are interpreted relative to neutral.

In [ ]:
def ordered_levels(data: pd.DataFrame, factor: str) -> list:
    template_order = (
        data[["template_id", factor]]
        .drop_duplicates()
        .sort_values("template_id")[factor]
        .dropna()
        .tolist()
    )
    seen = []
    for level in template_order:
        if level not in seen:
            seen.append(level)
    return seen


def paired_effects(data: pd.DataFrame, factor: str, metric: str, reference: object | None = None) -> pd.DataFrame:
    levels = ordered_levels(data, factor)
    if not levels:
        return pd.DataFrame()
    if reference is None:
        reference = levels[0]
    grouped = data.groupby(["model_short", "problem_id", factor], dropna=False)[metric].mean().reset_index()
    wide = grouped.pivot_table(index=["model_short", "problem_id"], columns=factor, values=metric, aggfunc="mean")
    rows = []
    for level in levels:
        if level == reference or level not in wide.columns or reference not in wide.columns:
            continue
        delta = (wide[level] - wide[reference]).dropna()
        n = len(delta)
        mean = delta.mean() if n else np.nan
        se = delta.std(ddof=1) / math.sqrt(n) if n > 1 else np.nan
        rows.append({
            "factor": factor, "metric": metric, "reference": reference, "level": level,
            "mean_delta": mean, "ci_low": mean - 1.96 * se if n > 1 else np.nan,
            "ci_high": mean + 1.96 * se if n > 1 else np.nan, "n_pairs": n,
        })
    return pd.DataFrame(rows)

paired_tables = []
for factor in technique_factors:
    for metric in summary_metrics:
        paired_tables.append(paired_effects(df, factor, metric))
paired_summary = pd.concat(paired_tables, ignore_index=True)
paired_summary.to_csv(OUTPUT_DIR / "paired_prompt_technique_effects.csv", index=False)
paired_summary[paired_summary["factor"].eq("Tone") & paired_summary["metric"].isin(["passed_int", "tests_fraction", "completion_tokens", "likely_code_only"])].style.format({"mean_delta": "{:+.4f}", "ci_low": "{:+.4f}", "ci_high": "{:+.4f}"})

## 8. Fixed-Effect Regression for Prompt Factors

This reruns the original fixed-effect regression:

```text
outcome ~ Tone + Role_framing + Format + model fixed effects + problem fixed effects
```

This controls for model ability and problem difficulty.

In [ ]:
def fit_fixed_effect(metric: str):
    factors = [f"C({factor})" for factor in technique_factors]
    rhs = " + ".join([*factors, "C(model_short)", "C(problem_id)"])
    formula = f"{metric} ~ {rhs}"
    model_data = df[[metric, "model_short", "problem_id", *technique_factors]].dropna()
    result = smf.ols(formula, data=model_data).fit(cov_type="HC3")
    return result, formula

regression_metrics = ["passed_int", "tests_fraction", "duration_seconds", "completion_tokens", "likely_code_only", "extra_text_ratio", "invalid_python_after_fix"]
regression_rows = []
regression_formulas = []
for metric in regression_metrics:
    result, formula = fit_fixed_effect(metric)
    params = result.params.rename("coef").to_frame()
    params["std_err"] = result.bse
    params["p_value"] = result.pvalues
    params["metric"] = metric
    params["term"] = params.index
    regression_rows.append(params.reset_index(drop=True))
    regression_formulas.append({"metric": metric, "formula": formula, "nobs": result.nobs, "rsquared": result.rsquared})
regression_table = pd.concat(regression_rows, ignore_index=True)
technique_terms = regression_table[regression_table["term"].str.contains("Tone|Role_framing|Format|Pressure", regex=True, na=False)].copy()
technique_terms.to_csv(OUTPUT_DIR / "fixed_effect_prompt_technique_models.csv", index=False)
pd.DataFrame(regression_formulas).to_csv(OUTPUT_DIR / "fixed_effect_prompt_technique_formulas.csv", index=False)
technique_terms.style.format({"coef": "{:+.4f}", "std_err": "{:.4f}", "p_value": "{:.4f}"})

## 9. Deep Behavior-Control Analysis

This is the professor's requested analysis.

We compare the baseline prompt model to models that add behavior controls:

- output length: `log_completion_tokens`
- code-only compliance: `likely_code_only`
- parsing/validity failure: `invalid_python_after_fix`
- runtime: `log_duration_seconds`

If the neutral-tone effect disappears after these controls, it suggests that tone works mainly by changing output behavior.

In [ ]:
def coef_rows(result, model_name: str, outcome: str) -> pd.DataFrame:
    rows = result.params.rename("coef").to_frame()
    rows["std_err"] = result.bse
    rows["p_value"] = result.pvalues
    rows["model_spec"] = model_name
    rows["outcome"] = outcome
    rows["term"] = rows.index
    return rows.reset_index(drop=True)


def fit_ols_behavior(outcome: str, controls: list[str], spec_name: str):
    prompt_terms = "C(Tone) + C(Role_framing) + C(Format)"
    fixed_effects = "C(model_short) + C(problem_id)"
    formula = f"{outcome} ~ " + " + ".join([prompt_terms, fixed_effects, *controls])
    model_cols = [outcome, "Tone", "Role_framing", "Format", "model_short", "problem_id", *controls]
    model_data = df[model_cols].dropna()
    result = smf.ols(formula, data=model_data).fit(cov_type="HC3")
    return result, formula

control_sets = {
    "M0_prompt_only": [],
    "M1_length": ["log_completion_tokens"],
    "M2_code_only": ["likely_code_only"],
    "M3_invalid_python": ["invalid_python_after_fix"],
    "M4_runtime": ["log_duration_seconds"],
    "M5_all_behavior_controls": ["log_completion_tokens", "likely_code_only", "invalid_python_after_fix", "log_duration_seconds"],
}

behavior_rows = []
behavior_formulas = []
for outcome in ["passed_int", "tests_fraction"]:
    for spec_name, controls in control_sets.items():
        result, formula = fit_ols_behavior(outcome, controls, spec_name)
        behavior_rows.append(coef_rows(result, spec_name, outcome))
        behavior_formulas.append({"outcome": outcome, "model_spec": spec_name, "formula": formula, "nobs": result.nobs, "rsquared": result.rsquared})
behavior_ols = pd.concat(behavior_rows, ignore_index=True)
behavior_ols.to_csv(OUTPUT_DIR / "behavior_control_ols_models.csv", index=False)
pd.DataFrame(behavior_formulas).to_csv(OUTPUT_DIR / "behavior_control_ols_formulas.csv", index=False)

focus_terms = ["C(Tone)[T.neutral]", "C(Tone)[T.polite]", "log_completion_tokens", "likely_code_only", "invalid_python_after_fix", "log_duration_seconds"]
focus_ols = behavior_ols[behavior_ols["term"].isin(focus_terms)].copy()
focus_ols.to_csv(OUTPUT_DIR / "focus_ols_terms.csv", index=False)
focus_ols[["outcome", "model_spec", "term", "coef", "std_err", "p_value"]].style.format({"coef": "{:+.4f}", "std_err": "{:.4f}", "p_value": "{:.4f}"})

## 10. Logistic Robustness Check

Because `passed_int` is binary, this reruns the behavior-control models with a binomial GLM/logit link.

In [ ]:
def fit_glm_binomial(controls: list[str], spec_name: str):
    prompt_terms = "C(Tone) + C(Role_framing) + C(Format)"
    fixed_effects = "C(model_short) + C(problem_id)"
    formula = "passed_int ~ " + " + ".join([prompt_terms, fixed_effects, *controls])
    model_cols = ["passed_int", "Tone", "Role_framing", "Format", "model_short", "problem_id", *controls]
    model_data = df[model_cols].dropna()
    result = smf.glm(formula=formula, data=model_data, family=sm.families.Binomial()).fit(cov_type="HC3")
    return result, formula

glm_rows = []
glm_formulas = []
for spec_name, controls in control_sets.items():
    result, formula = fit_glm_binomial(controls, spec_name)
    glm_rows.append(coef_rows(result, spec_name, "passed_int_glm_binomial_logit"))
    glm_formulas.append({"outcome": "passed_int", "model_spec": spec_name, "formula": formula, "nobs": result.nobs, "aic": result.aic})
glm_results = pd.concat(glm_rows, ignore_index=True)
glm_results.to_csv(OUTPUT_DIR / "behavior_control_glm_logit_models.csv", index=False)
pd.DataFrame(glm_formulas).to_csv(OUTPUT_DIR / "behavior_control_glm_formulas.csv", index=False)
focus_glm = glm_results[glm_results["term"].isin(focus_terms)].copy()
focus_glm.to_csv(OUTPUT_DIR / "focus_glm_terms.csv", index=False)
focus_glm[["model_spec", "term", "coef", "std_err", "p_value"]].style.format({"coef": "{:+.4f}", "std_err": "{:.4f}", "p_value": "{:.4f}"})

## 11. Model-Specific and Difficulty Analyses

These checks prevent overclaiming.

They answer:

- Is neutral tone best for every model?
- Does tone help equally on hard, medium, and easy problems?

In [ ]:
model_tone = (
    df.groupby(["model_short", "Tone"])
    .agg(
        runs=("task_id", "size"),
        pass_rate=("passed_int", "mean"),
        tests_fraction=("tests_fraction", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_duration_seconds=("duration_seconds", "mean"),
        code_only_rate=("likely_code_only", "mean"),
        invalid_python_rate=("invalid_python_after_fix", "mean"),
        runtime_failure_rate=("runtime_failure", "mean"),
    )
    .reset_index()
    .sort_values(["model_short", "tests_fraction"], ascending=[True, False])
)
model_tone.to_csv(OUTPUT_DIR / "model_by_tone_behavior_summary.csv", index=False)
model_tone.style.format({"pass_rate": "{:.1%}", "tests_fraction": "{:.1%}", "avg_completion_tokens": "{:.0f}", "avg_duration_seconds": "{:.2f}", "code_only_rate": "{:.1%}", "invalid_python_rate": "{:.1%}", "runtime_failure_rate": "{:.1%}"})

In [ ]:
problem_difficulty = (
    df.groupby("problem_id")
    .agg(problem_mean_tests_fraction=("tests_fraction", "mean"), problem_pass_rate=("passed_int", "mean"))
    .reset_index()
)
problem_difficulty["difficulty_band"] = pd.qcut(problem_difficulty["problem_mean_tests_fraction"], q=3, labels=["hard", "medium", "easy"], duplicates="drop")
df = df.drop(columns=["difficulty_band"], errors="ignore").merge(problem_difficulty[["problem_id", "difficulty_band"]], on="problem_id", how="left")

difficulty_tone = (
    df.groupby(["difficulty_band", "Tone"], observed=False)
    .agg(
        runs=("task_id", "size"),
        pass_rate=("passed_int", "mean"),
        tests_fraction=("tests_fraction", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_duration_seconds=("duration_seconds", "mean"),
        code_only_rate=("likely_code_only", "mean"),
        invalid_python_rate=("invalid_python_after_fix", "mean"),
        runtime_failure_rate=("runtime_failure", "mean"),
    )
    .reset_index()
)
difficulty_tone.to_csv(OUTPUT_DIR / "difficulty_by_tone_behavior_summary.csv", index=False)
difficulty_tone.style.format({"pass_rate": "{:.1%}", "tests_fraction": "{:.1%}", "avg_completion_tokens": "{:.0f}", "avg_duration_seconds": "{:.2f}", "code_only_rate": "{:.1%}", "invalid_python_rate": "{:.1%}", "runtime_failure_rate": "{:.1%}"})

## 12. Error Categories and Behavior Correlations

These tables help explain the mechanism behind tone effects.

In [ ]:
error_summary = (
    df.groupby(["Tone", "error_category"])
    .size()
    .rename("runs")
    .reset_index()
)
error_summary["rate"] = error_summary["runs"] / error_summary.groupby("Tone")["runs"].transform("sum")
error_summary = error_summary.sort_values(["Tone", "runs"], ascending=[True, False])
error_summary.to_csv(OUTPUT_DIR / "tone_error_category_summary.csv", index=False)
error_summary.style.format({"rate": "{:.1%}"})

In [ ]:
corr_cols = ["passed_int", "tests_fraction", "completion_tokens", "duration_seconds", "likely_code_only", "invalid_python_after_fix", "runtime_failure", "extra_text_ratio"]
correlations = df[corr_cols].corr(method="pearson")
correlations.to_csv(OUTPUT_DIR / "behavior_metric_correlations.csv")
correlations.style.format("{:.3f}").background_gradient(cmap="coolwarm", vmin=-1, vmax=1)

## 13. Best and Worst Templates

This reruns the original template-level summary using fixed-extractor results.

In [ ]:
template_summary = (
    df.groupby(["template_id", *technique_factors])
    .agg(
        runs=("task_id", "size"),
        models=("model_short", "nunique"),
        problems=("problem_id", "nunique"),
        pass_rate=("passed_int", "mean"),
        tests_fraction=("tests_fraction", "mean"),
        avg_duration_seconds=("duration_seconds", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        code_only_rate=("likely_code_only", "mean"),
        invalid_python_rate=("invalid_python_after_fix", "mean"),
    )
    .reset_index()
    .sort_values(["tests_fraction", "pass_rate"], ascending=False)
)
template_summary.to_csv(OUTPUT_DIR / "prompt_template_summary.csv", index=False)
print("Best templates")
display(template_summary.head(8).style.format({"pass_rate": "{:.1%}", "tests_fraction": "{:.1%}", "avg_duration_seconds": "{:.2f}", "avg_completion_tokens": "{:.0f}", "code_only_rate": "{:.1%}", "invalid_python_rate": "{:.1%}"}))
print("Worst templates")
display(template_summary.tail(8).style.format({"pass_rate": "{:.1%}", "tests_fraction": "{:.1%}", "avg_duration_seconds": "{:.2f}", "avg_completion_tokens": "{:.0f}", "code_only_rate": "{:.1%}", "invalid_python_rate": "{:.1%}"}))

## 14. Write Updated Report

This cell writes a compact markdown report from the corrected results.

In [ ]:
def pct(x):
    return f"{100*x:.1f}%"


def pp(x):
    return f"{100*x:+.1f} pp"


def get_factor_mean(factor, level, metric):
    row = factor_summary[(factor_summary["factor"].eq(factor)) & (factor_summary[factor].eq(level)) & (factor_summary["metric"].eq(metric))]
    return float(row["mean"].iloc[0])


def get_paired(factor, level, metric):
    row = paired_summary[(paired_summary["factor"].eq(factor)) & (paired_summary["level"].eq(level)) & (paired_summary["metric"].eq(metric))]
    return float(row["mean_delta"].iloc[0]) if len(row) else float("nan")


def get_focus(outcome, spec, term, col="coef"):
    row = focus_ols[(focus_ols["outcome"].eq(outcome)) & (focus_ols["model_spec"].eq(spec)) & (focus_ols["term"].eq(term))]
    return float(row[col].iloc[0]) if len(row) else float("nan")

neutral_pass = get_factor_mean("Tone", "neutral", "passed_int")
polite_pass = get_factor_mean("Tone", "polite", "passed_int")
aggr_pass = get_factor_mean("Tone", "aggressive", "passed_int")
neutral_tests = get_factor_mean("Tone", "neutral", "tests_fraction")
polite_tests = get_factor_mean("Tone", "polite", "tests_fraction")
aggr_tests = get_factor_mean("Tone", "aggressive", "tests_fraction")
neutral_tokens = get_factor_mean("Tone", "neutral", "completion_tokens")
polite_tokens = get_factor_mean("Tone", "polite", "completion_tokens")
aggr_tokens = get_factor_mean("Tone", "aggressive", "completion_tokens")
neutral_codeonly = get_factor_mean("Tone", "neutral", "likely_code_only")
polite_codeonly = get_factor_mean("Tone", "polite", "likely_code_only")
aggr_codeonly = get_factor_mean("Tone", "aggressive", "likely_code_only")

report = f"""# Fixed-Extractor Prompt Technique Effects Report

Based on `fixed_extractor_analysis.executed.ipynb` and the corrected `*_fixextractor.jsonl` result files.

## Executive Summary

After fixing the code extractor, the overall conclusion remains the same but is cleaner: **tone affects both model behavior and performance**. Neutral prompts still perform best overall, but the deeper behavior-control analysis shows that the apparent neutral-tone advantage is largely explained by output behavior: shorter outputs, faster runtime, and much higher code-only compliance.

The extractor fix mainly affected `llama3.1-8b`: its exact pass count increased from 35/900 to 48/900 because many plain Python responses had previously been missed by the markdown-fence-only extractor.

## Data Analyzed

- Result files: `results/gemma2-9b_fixextractor.jsonl`, `results/llama3.1-8b_fixextractor.jsonl`, `results/qwen2.5-7b_fixextractor.jsonl`
- Models: 3
- Problems: 50
- Prompt templates: 18
- Total runs: 2,700
- Prompt factors analyzed: `Tone`, `Role_framing`, `Format`
- Output directory: `analysis_outputs/fixed_extractor/`

## Extractor Fix Impact

| Model | Original passes | Fixed passes | Changed extractions | Changed pass/fail |
|---|---:|---:|---:|---:|
"""
for _, r in extractor_impact.sort_values("model_short").iterrows():
    report += f"| {r['model_short']} | {int(r['original_passes'])}/900 | {int(r['fixed_passes'])}/900 | {int(r['changed_extraction'])} | {int(r['changed_pass'])} |\n"

report += f"""

The fix confirms that the original extraction failure variable should be interpreted as a pipeline compatibility issue, not always a true model failure. The fixed evaluator now treats unfenced plain Python as candidate code and rejects non-Python responses with `Response is not valid Python`.

## Corrected Model Performance

| Model | Runs | Pass rate | Avg. tests passed | Avg. duration | Avg. completion tokens | Code-only rate |
|---|---:|---:|---:|---:|---:|---:|
"""
for _, r in model_summary.iterrows():
    report += f"| {r['model_short']} | {int(r['runs'])} | {pct(r['pass_rate'])} | {pct(r['avg_tests_fraction'])} | {r['avg_duration_seconds']:.2f}s | {r['avg_completion_tokens']:.0f} | {pct(r['code_only_rate'])} |\n"

report += f"""

## Main Finding: Neutral Tone Still Performs Best Overall

| Tone | Runs | Pass rate | Avg. tests passed | Avg. completion tokens | Avg. duration | Code-only rate |
|---|---:|---:|---:|---:|---:|---:|
| neutral | 900 | {pct(neutral_pass)} | {pct(neutral_tests)} | {neutral_tokens:.0f} | {get_factor_mean('Tone', 'neutral', 'duration_seconds'):.2f}s | {pct(neutral_codeonly)} |
| polite | 900 | {pct(polite_pass)} | {pct(polite_tests)} | {polite_tokens:.0f} | {get_factor_mean('Tone', 'polite', 'duration_seconds'):.2f}s | {pct(polite_codeonly)} |
| aggressive | 900 | {pct(aggr_pass)} | {pct(aggr_tests)} | {aggr_tokens:.0f} | {get_factor_mean('Tone', 'aggressive', 'duration_seconds'):.2f}s | {pct(aggr_codeonly)} |

Paired comparisons versus neutral show:

| Comparison | Pass-rate delta | Tests-fraction delta | Completion-token delta | Code-only-rate delta |
|---|---:|---:|---:|---:|
| polite vs neutral | {pp(get_paired('Tone', 'polite', 'passed_int'))} | {pp(get_paired('Tone', 'polite', 'tests_fraction'))} | {get_paired('Tone', 'polite', 'completion_tokens'):+.0f} | {pp(get_paired('Tone', 'polite', 'likely_code_only'))} |
| aggressive vs neutral | {pp(get_paired('Tone', 'aggressive', 'passed_int'))} | {pp(get_paired('Tone', 'aggressive', 'tests_fraction'))} | {get_paired('Tone', 'aggressive', 'completion_tokens'):+.0f} | {pp(get_paired('Tone', 'aggressive', 'likely_code_only'))} |

## Deep Behavior-Control Result

With aggressive tone as the regression reference category, the baseline fixed-effect model estimates neutral tone as:

- pass/fail: {pp(get_focus('passed_int', 'M0_prompt_only', 'C(Tone)[T.neutral]'))}, p = {get_focus('passed_int', 'M0_prompt_only', 'C(Tone)[T.neutral]', 'p_value'):.4f}
- tests fraction: {pp(get_focus('tests_fraction', 'M0_prompt_only', 'C(Tone)[T.neutral]'))}, p = {get_focus('tests_fraction', 'M0_prompt_only', 'C(Tone)[T.neutral]', 'p_value'):.4f}

After adding behavior controls, the neutral-tone coefficient becomes non-significant:

| Outcome | Model | Neutral coefficient | p-value |
|---|---|---:|---:|
| pass/fail | prompt only | {pp(get_focus('passed_int', 'M0_prompt_only', 'C(Tone)[T.neutral]'))} | {get_focus('passed_int', 'M0_prompt_only', 'C(Tone)[T.neutral]', 'p_value'):.4f} |
| pass/fail | + output length | {pp(get_focus('passed_int', 'M1_length', 'C(Tone)[T.neutral]'))} | {get_focus('passed_int', 'M1_length', 'C(Tone)[T.neutral]', 'p_value'):.4f} |
| pass/fail | + code-only | {pp(get_focus('passed_int', 'M2_code_only', 'C(Tone)[T.neutral]'))} | {get_focus('passed_int', 'M2_code_only', 'C(Tone)[T.neutral]', 'p_value'):.4f} |
| pass/fail | + runtime | {pp(get_focus('passed_int', 'M4_runtime', 'C(Tone)[T.neutral]'))} | {get_focus('passed_int', 'M4_runtime', 'C(Tone)[T.neutral]', 'p_value'):.4f} |
| pass/fail | all behavior controls | {pp(get_focus('passed_int', 'M5_all_behavior_controls', 'C(Tone)[T.neutral]'))} | {get_focus('passed_int', 'M5_all_behavior_controls', 'C(Tone)[T.neutral]', 'p_value'):.4f} |
| tests fraction | prompt only | {pp(get_focus('tests_fraction', 'M0_prompt_only', 'C(Tone)[T.neutral]'))} | {get_focus('tests_fraction', 'M0_prompt_only', 'C(Tone)[T.neutral]', 'p_value'):.4f} |
| tests fraction | + output length | {pp(get_focus('tests_fraction', 'M1_length', 'C(Tone)[T.neutral]'))} | {get_focus('tests_fraction', 'M1_length', 'C(Tone)[T.neutral]', 'p_value'):.4f} |
| tests fraction | + code-only | {pp(get_focus('tests_fraction', 'M2_code_only', 'C(Tone)[T.neutral]'))} | {get_focus('tests_fraction', 'M2_code_only', 'C(Tone)[T.neutral]', 'p_value'):.4f} |
| tests fraction | + runtime | {pp(get_focus('tests_fraction', 'M4_runtime', 'C(Tone)[T.neutral]'))} | {get_focus('tests_fraction', 'M4_runtime', 'C(Tone)[T.neutral]', 'p_value'):.4f} |
| tests fraction | all behavior controls | {pp(get_focus('tests_fraction', 'M5_all_behavior_controls', 'C(Tone)[T.neutral]'))} | {get_focus('tests_fraction', 'M5_all_behavior_controls', 'C(Tone)[T.neutral]', 'p_value'):.4f} |

This supports the mediation-style interpretation: **neutral prompts perform better largely because they produce shorter, cleaner, more code-focused outputs.**

## Model-Specific Caveat

Neutral tone is strongest overall, but the effect is not identical across models. After fixing extraction, Llama improves, but the overall tone mechanism still differs by model family. The paper should avoid claiming that neutral tone universally improves every model in every setting.

## Interpretation

The strongest updated interpretation is:

> Prompt tone affects downstream coding performance mostly by changing output behavior. Neutral prompts lead to shorter, faster, more code-only responses, which fit the evaluator better. Polite and aggressive prompts encourage longer explanatory responses, which increases latency and reduces code-only compliance; these behavioral shifts explain much of the observed performance gap.

## Caveats

- Behavior controls are post-treatment variables, so they explain mechanism but should not be interpreted as fully causal controls.
- Completion tokens and runtime are highly correlated, so their individual coefficients should be interpreted carefully when included together.
- The fixed extractor reduces false `No code extracted` failures, but future experiments should use the fixed extractor from the start.
- Exact pass rates remain low, so `tests_fraction` is still the more informative performance metric.

## Recommended Paper Framing

Frame the paper as:

> a controlled study of how prompt tone changes output behavior, compliance, latency, verbosity, and downstream task performance.

The corrected data strengthens the paper because it removes a known evaluator artifact while preserving the main behavioral explanation.
"""

REPORT_PATH.write_text(report, encoding="utf-8")
print(f"Wrote {REPORT_PATH}")
print(report[:1600])